# Code Review Agent Exploration

Run the cells from top to bottom. The final review cell requires `OPENAI_API_KEY` in `.env`.

## 1. Import the project code

In [2]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# This notebook is inside explorations/, so import from its parent folder.
project_folder = Path.cwd().parent
if str(project_folder) not in sys.path:
    sys.path.insert(0, str(project_folder))

from agent import (
    MAX_CODE_LENGTH,
    build_review_messages,
    read_code_file,
    review_code,
    validate_code,
)

load_dotenv(project_folder / '.env')
print('Project code imported successfully.')

Project code imported successfully.


In [5]:
# Confirm the key exists without printing its secret value.
status = 'configured' if os.getenv('OPENAI_API_KEY') else 'missing'
print(f'OPENAI_API_KEY: {status}')

OPENAI_API_KEY: configured


## 2. Validate source code

The agent trims whitespace, rejects empty input, and limits large requests before calling OpenAI.

In [7]:
# This sample contains an intentional division-by-zero risk.
sample_code = '''
def divide(left, right):
    return left / right
'''

clean_code = validate_code(sample_code)
print(clean_code)
print(f'Characters: {len(clean_code)} / {MAX_CODE_LENGTH:,}')

def divide(left, right):
    return left / right
Characters: 48 / 100,000


In [8]:
# Empty input is rejected before any API call.
try:
    validate_code('   ')
except ValueError as error:
    print(f'Validation error: {error}')

Validation error: Code to review cannot be empty.


## 3. Build the review prompt

The prompt has a system instruction and a user message containing the source code.

In [10]:
messages = build_review_messages(clean_code, 'python')
print(f'Messages created: {len(messages)}')
print(messages[1].content)

Messages created: 2
Review this python code:

```python
def divide(left, right):
    return left / right
```


## 4. Read a source file

`read_code_file()` reads UTF-8 code and applies the same validation rules.

In [12]:
example_path = Path('example_to_review.py')
example_path.write_text(sample_code, encoding='utf-8')
print(read_code_file(str(example_path)))
example_path.unlink()  # Remove the temporary example.

def divide(left, right):
    return left / right


## 5. Run a complete review

Run this cell only when `OPENAI_API_KEY` is configured. It calls `gpt-4o` and prints the Markdown report.

In [13]:
if not os.getenv('OPENAI_API_KEY'):
    print('Skipped: OPENAI_API_KEY is missing.')
else:
    review = review_code(sample_code, language='python')
    print(review)

# Code Review

## 1. Bugs & Correctness
- **Division by Zero**: The function does not handle the case where `right` is zero, which will raise a `ZeroDivisionError`. This should be handled to prevent the program from crashing.
- **Type Checking**: The function assumes that both `left` and `right` are numbers. If non-numeric types are passed, a `TypeError` will be raised. Consider adding type checks or using type hints to clarify expected input types.

## 2. Security Issues
- **Injection Risks**: There are no apparent injection risks in this simple function.
- **Secrets Exposure**: The function does not handle any sensitive data, so there are no concerns about secrets exposure.

## 3. Performance
- **Inefficiencies**: The function performs a single division operation, which is efficient.
- **Unnecessary Computation**: There are no unnecessary computations in this function.

## 4. Code Style
- **PEP 8 Violations**: The code is very short and adheres to PEP 8 guidelines.
- **Naming Convent

## 6. Run the Streamlit UI

The UI in `app.py` uses the same `review_code()` function. From Command Prompt in the agent folder, run:

```cmd
python -m streamlit run app.py
```